Question 1: Custom Feature Transformation Using FunctionTransformer
A financial analytics team has collected customer income data. The income distribution is highly right-skewed, and large values dominate the original scale.

import numpy as np
import pandas as pd

df = pd.DataFrame({
    "Customer_ID": range(1, 11),
    "Income": [15000, 18000, 22000, 25000, 30000,
               35000, 42000, 55000, 120000, 500000],
    "Monthly_Expenses": [8000, 9000, 11000, 12000, 15000,
                         17000, 20000, 25000, 45000, 180000]
})
A. Use FunctionTransformer with np.log1p to transform both numerical features. Compare the original and transformed means, ranges, and standard deviations. Explain mathematically why the transformation compresses large values more strongly than small values.

B. Create a scikit-learn pipeline that applies the logarithmic transformation before fitting a LinearRegression model to predict monthly expenses from income. Compare its performance against a model trained on untransformed income using a train-test split and an appropriate regression metric. Interpret the results without assuming the transformation must perform better.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer

df = pd.DataFrame({
    "Customer_ID": range(1, 11),
    "Income": [15000, 18000, 22000, 25000, 30000,
        35000, 42000, 55000, 120000, 500000],
    "Monthly_Expenses": [8000, 9000, 11000, 12000, 15000,
        17000, 20000, 25000, 45000, 180000]})

features = ["Income", "Monthly_Expenses"]

log_transformer = FunctionTransformer(
    np.log1p,
    feature_names_out="one-to-one"
)

df_log = df.copy()
df_log[features] = log_transformer.fit_transform(df[features])

comparison = pd.DataFrame({
    "Original Mean": df[features].mean(),
    "Log Mean": df_log[features].mean(),
    "Original Range": (
        df[features].max() - df[features].min()
    ),
    "Log Range": (
        df_log[features].max() - df_log[features].min()
    ),
    "Original Std": df[features].std(),
    "Log Std": df_log[features].std()
})

print("Original Data:")
print(df[features])

print("\nLog Transformed Data:")
print(df_log[features].round(3))

print("\nStatistical Comparison:")
print(comparison.round(3))


Original Data:
   Income  Monthly_Expenses
0   15000              8000
1   18000              9000
2   22000             11000
3   25000             12000
4   30000             15000
5   35000             17000
6   42000             20000
7   55000             25000
8  120000             45000
9  500000            180000

Log Transformed Data:
   Income  Monthly_Expenses
0   9.616             8.987
1   9.798             9.105
2   9.999             9.306
3  10.127             9.393
4  10.309             9.616
5  10.463             9.741
6  10.645             9.904
7  10.915            10.127
8  11.695            10.714
9  13.122            12.101

Statistical Comparison:
                  Original Mean  Log Mean  Original Range  Log Range  \
Income                  86200.0    10.669          485000      3.506   
Monthly_Expenses        34200.0     9.899          172000      3.113   

                  Original Std  Log Std  
Income              148573.813    1.050  
Monthly_Expenses    

B. Create a scikit-learn pipeline that applies the logarithmic transformation before fitting a LinearRegression model to predict monthly expenses from income. Compare its performance against a model trained on untransformed income using a train-test split and an appropriate regression metric. Interpret the results without assuming the transformation must perform better.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df = pd.DataFrame({
    "Customer_ID": range(1, 11),
    "Income": [15000, 18000, 22000, 25000, 30000,
        35000, 42000, 55000, 120000, 500000],
    "Monthly_Expenses": [8000, 9000, 11000, 12000, 15000,
        17000, 20000, 25000, 45000, 180000]})
X = df[["Income"]]
y = df["Monthly_Expenses"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

original_model = Pipeline([
    ("regression", LinearRegression())
])

original_model.fit(X_train, y_train)
pred_original = original_model.predict(X_test)

mae_original = mean_absolute_error(
    y_test, pred_original
)

log_model = Pipeline([
    ("log_transform", FunctionTransformer(np.log1p)),
    ("regression", LinearRegression())
])

log_model.fit(X_train, y_train)
pred_log = log_model.predict(X_test)

mae_log = mean_absolute_error(
    y_test, pred_log
)
print("Actual Expenses:")
print(y_test.to_list())

print("\nOriginal Model Predictions:")
print(np.round(pred_original, 2))

print("\nLog Model Predictions:")
print(np.round(pred_log, 2))

print("\nOriginal Model MAE:", round(mae_original, 2))
print("Log Model MAE:", round(mae_log, 2))

if mae_log < mae_original:
    print("Log transformation performed better on this test set.")
elif mae_log > mae_original:
    print("Original income performed better on this test set.")
else:
    print("Both models have equal MAE on this test set.")


Actual Expenses:
[45000, 9000, 17000]

Original Model Predictions:
[46313.41 10381.13 16369.85]

Log Model Predictions:
[91507.24 -6770.61 27677.08]

Original Model MAE: 1108.23
Log Model MAE: 24318.31
Original income performed better on this test set.


Question 2: Designing and Validating Custom Transformations
An e-commerce company wants to transform sales and transaction features before training a machine learning model.

df = pd.DataFrame({
    "Sales": [100, 150, 200, 250, 400, 600, 900, 1500, 3000, 8000],
    "Transaction_Count": [1, 2, 2, 3, 4, 5, 7, 10, 15, 30],
    "Discount_Percent": [0, 5, 10, 15, 20, 5, 10, 25, 30, 40],
    "Revenue": [120, 170, 230, 280, 450, 650, 950, 1600, 3200, 8500]
})
A. Use FunctionTransformer to create and apply a custom function that converts sales into thousands of units. Apply a separate square-root transformation to Transaction_Count. Compare the transformed values with the original values and explain when each transformation could be useful.


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import FunctionTransformer

df = pd.DataFrame({
    "Sales": [100, 150, 200, 250, 400, 600, 900, 1500, 3000, 8000],
    "Transaction_Count": [1, 2, 2, 3, 4, 5, 7, 10, 15, 30],
    "Discount_Percent": [0, 5, 10, 15, 20, 5, 10, 25, 30, 40],
    "Revenue": [120, 170, 230, 280, 450, 650, 950, 1600, 3200, 8500]
})

def sales_in_thousands(x):
    return x / 1000

sales_transformer = FunctionTransformer(sales_in_thousands)
sqrt_transformer = FunctionTransformer(np.sqrt)

df_transformed = df.copy()

df_transformed["Sales"] = sales_transformer.transform(
    df[["Sales"]]
)

df_transformed["Transaction_Count"] = sqrt_transformer.transform(
    df[["Transaction_Count"]]
)

print("Original Values:")
print(df[["Sales", "Transaction_Count"]])

print("\nTransformed Values:")
print(df_transformed[["Sales", "Transaction_Count"]].round(3))


Original Values:
   Sales  Transaction_Count
0    100                  1
1    150                  2
2    200                  2
3    250                  3
4    400                  4
5    600                  5
6    900                  7
7   1500                 10
8   3000                 15
9   8000                 30

Transformed Values:
   Sales  Transaction_Count
0   0.10              1.000
1   0.15              1.414
2   0.20              1.414
3   0.25              1.732
4   0.40              2.000
5   0.60              2.236
6   0.90              2.646
7   1.50              3.162
8   3.00              3.873
9   8.00              5.477



B. Implement a reusable custom transformation that clips a numerical feature to a specified lower and upper bound. Apply it to Sales, compare the results for at least two different clipping ranges, and demonstrate how clipping changes the mean, standard deviation, and extreme values. Explain why clipping should be based on domain knowledge rather than arbitrary limits.

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import FunctionTransformer

df = pd.DataFrame({
    "Sales": [100, 150, 200, 250, 400, 600, 900, 1500, 3000, 8000],
    "Transaction_Count": [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
})

print("Original DataFrame:")
print(df)

def sales_in_thousands(x):
    return x / 1000

sales_transformer = FunctionTransformer(sales_in_thousands)

df["Sales_in_Thousands"] = sales_transformer.transform(
    df[["Sales"]].to_numpy()
).flatten()

sqrt_transformer = FunctionTransformer(np.sqrt)

df["Transaction_Sqrt"] = sqrt_transformer.transform(
    df[["Transaction_Count"]].to_numpy()
).flatten()

def clip_values(x, lower, upper):
    return np.clip(x, lower, upper)

clip_3000 = FunctionTransformer(
    clip_values,
    kw_args={"lower": 100, "upper": 3000}
)

clip_1500 = FunctionTransformer(
    clip_values,
    kw_args={"lower": 200, "upper": 1500}
)

df["Sales_Clip_3000"] = clip_3000.transform(
    df[["Sales"]].to_numpy()
).flatten()

df["Sales_Clip_1500"] = clip_1500.transform(
    df[["Sales"]].to_numpy()
).flatten()

print("\nDataFrame After Transformations:")
print(df)

columns = ["Sales","Sales_Clip_3000","Sales_Clip_1500"]

print("\nSales Statistics Comparison:")
print(df[columns].agg(["mean", "std", "min", "max"]).round(2))


Original DataFrame:
   Sales  Transaction_Count
0    100                  1
1    150                  4
2    200                  9
3    250                 16
4    400                 25
5    600                 36
6    900                 49
7   1500                 64
8   3000                 81
9   8000                100

DataFrame After Transformations:
   Sales  Transaction_Count  Sales_in_Thousands  Transaction_Sqrt  \
0    100                  1                0.10               1.0   
1    150                  4                0.15               2.0   
2    200                  9                0.20               3.0   
3    250                 16                0.25               4.0   
4    400                 25                0.40               5.0   
5    600                 36                0.60               6.0   
6    900                 49                0.90               7.0   
7   1500                 64                1.50               8.0   
8   3000         

Question 3: PowerTransformer and Distribution Analysis
A bank is studying customer financial data. Several features are strongly skewed, and the team wants to investigate whether power transformations make them more symmetric.

import numpy as np
import pandas as pd

df = pd.DataFrame({
    "Annual_Income": [20000, 22000, 25000, 28000, 32000,
                      40000, 50000, 70000, 150000, 600000],
    "Loan_Amount": [5000, 7000, 9000, 12000, 15000,
                    20000, 30000, 50000, 100000, 400000],
    "Account_Balance": [-500, 0, 200, 500, 1000,
                        2500, 4000, 8000, 15000, 50000]
})
A. Apply PowerTransformer(method="yeo-johnson") to all three numerical features. Calculate and compare their skewness before and after transformation. Display the learned lambda values and explain why transformed data is not guaranteed to be perfectly normally distributed.



In [10]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PowerTransformer
df = pd.DataFrame({
    "Annual_Income": [20000, 22000, 25000, 28000, 32000,
                      40000, 50000, 70000, 150000, 600000],
    "Loan_Amount": [5000, 7000, 9000, 12000, 15000,
                    20000, 30000, 50000, 100000, 400000],
    "Account_Balance": [-500, 0, 200, 500, 1000,
                        2500, 4000, 8000, 15000, 50000]
})

features = ["Annual_Income", "Loan_Amount", "Account_Balance"]

print("Original Data:")
print(df)

skew_before = df[features].skew()

yeo = PowerTransformer(method="yeo-johnson")

transformed_data = yeo.fit_transform(df[features])

df_transformed = pd.DataFrame(
    transformed_data,
    columns=features
)

skew_after = df_transformed.skew()

comparison = pd.DataFrame({
    "Skewness_Before": skew_before,
    "Skewness_After": skew_after
})

print("\nSkewness Comparison:")
print(comparison.round(4))

lambda_values = pd.Series(
    yeo.lambdas_,
    index=features,
    name="Lambda"
)

print("\nLearned Lambda Values:")
print(lambda_values.round(4))

print("\nTransformed Data:")
print(df_transformed.round(4))


Original Data:
   Annual_Income  Loan_Amount  Account_Balance
0          20000         5000             -500
1          22000         7000                0
2          25000         9000              200
3          28000        12000              500
4          32000        15000             1000
5          40000        20000             2500
6          50000        30000             4000
7          70000        50000             8000
8         150000       100000            15000
9         600000       400000            50000

Skewness Comparison:
                 Skewness_Before  Skewness_After
Annual_Income             2.9096          0.4283
Loan_Amount               2.8604          0.1895
Account_Balance           2.6636          0.9304

Learned Lambda Values:
Annual_Income     -0.7816
Loan_Amount       -0.3554
Account_Balance    0.7416
Name: Lambda, dtype: float64

Transformed Data:
   Annual_Income  Loan_Amount  Account_Balance
0        -1.3798      -1.5985          -1.7908
1     

B. Apply PowerTransformer(method="box-cox") separately to the strictly positive Annual_Income and Loan_Amount features. Then attempt to apply Box-Cox to Account_Balance. Explain the validity of each operation and implement a safe preprocessing approach that selects a suitable transformation based on whether a feature contains zero or negative values.


In [11]:
positive_features = ["Annual_Income", "Loan_Amount"]
boxcox = PowerTransformer(method="box-cox")
boxcox_data = boxcox.fit_transform(df[positive_features])

df_boxcox = pd.DataFrame(
    boxcox_data,
    columns=["Annual_Income_BoxCox", "Loan_Amount_BoxCox"]
)

print("Box-Cox Transformed Data:")
print(df_boxcox.round(4))

print("\nBox-Cox Lambda Values:")
print(pd.Series(
    boxcox.lambdas_,
    index=positive_features
).round(4))

# B2. Attempt Box-Cox on Account_Balance
try:
    boxcox_balance = PowerTransformer(method="box-cox")
    boxcox_balance.fit_transform(df[["Account_Balance"]])
except ValueError as error:
    print("\nBox-Cox Error for Account_Balance:")
    print(error)

safe_results = {}

for column in df.columns:
    values = df[[column]]

    if (values > 0).all().all():
        transformer = PowerTransformer(method="box-cox")
        method = "Box-Cox"
    else:
        transformer = PowerTransformer(method="yeo-johnson")
        method = "Yeo-Johnson"

    safe_results[column] = transformer.fit_transform(
        values
    ).flatten()

    print(f"{column}: {method}")

safe_df = pd.DataFrame(safe_results)

print("\nSafe Preprocessed Data:")
print(safe_df.round(4))


Box-Cox Transformed Data:
   Annual_Income_BoxCox  Loan_Amount_BoxCox
0               -1.3798             -1.5986
1               -1.1351             -1.1188
2               -0.8342             -0.7961
3               -0.5915             -0.4603
4               -0.3317             -0.2226
5                0.0461              0.0574
6                0.3635              0.4065
7                0.7487              0.7804
8                1.3235              1.1904
9                1.7904              1.7616

Box-Cox Lambda Values:
Annual_Income   -0.7816
Loan_Amount     -0.3554
dtype: float64

Box-Cox Error for Account_Balance:
The Box-Cox transformation can only be applied to strictly positive data
Annual_Income: Box-Cox
Loan_Amount: Box-Cox
Account_Balance: Yeo-Johnson

Safe Preprocessed Data:
   Annual_Income  Loan_Amount  Account_Balance
0        -1.3798      -1.5986          -1.7908
1        -1.1351      -1.1188          -0.4355
2        -0.8342      -0.7961          -0.3894
3       